# Fused Layer Norm + ReLU without bitmask

This notebook shows how to compute forward Layer Norm (training) + clamped ReLU, then compute the backward equivalent (DReLU + DLN) by determinining dReLU activations by recomputing the y output with a prologue forward layer norm inference fusion. 

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NVIDIA/cudnn-frontend/blob/main/samples/python/25_layernorm_forward_training_and_backward_with_relu_bitmask.ipynb)

## Prerequisites and Setup
This notebook requires an NVIDIA GPU. If `nvidia-smi` fails, go to Runtime -> Change runtime type -> Hardware accelerator and confirm a GPU is selected.

In [ ]:
#get_ipython().system('nvidia-smi')

If running on Colab, you will need to install the cudnn python interface.

In [ ]:
# get_ipython().system('pip install nvidia-cudnn-cu12')
# get_ipython().system('pip install nvidia-cudnn-frontend')
# get_ipython().system('pip3 install --pre torch --index-url https://download.pytorch.org/whl/nightly/cu128')

## Overview

In the following, we will apply layer norm to a tensor of the following shape:

- Batch Size: 4
- Sequence Size: 1024
- Embedding Dimension: 768

Let's define these dimensions as constants:

In [1]:
import cudnn
import torch

torch.manual_seed(1)

handle = cudnn.create_handle()
print("Running with cudnn backend version:", cudnn.backend_version())

assert torch.cuda.is_available()
assert (
    cudnn.backend_version() >= 91800
), "dLN + dReLU fusion pattern with no bitmask is only supported cuDNN version 9.18.0 or above"

batch, seq_size, embedding_dim = 4, 1024, 768
dtype = torch.float32
# Epsilon is a small number to prevent division by 0.
epsilon_value = 1e-3
# Set clamped ReLU limits
lower_clip_val = 0
upper_clip_val = 6

Running with cudnn backend version: 91800


## Using Wrapper

#### LayerNorm ReLU Training Forward Pass

First, we define the input tensors

In [3]:
# random tensors as input
x_gpu = torch.randn(
    batch * seq_size,
    embedding_dim,
    1,
    1,
    device="cuda",
    dtype=dtype,
    requires_grad=True,
).to(memory_format=torch.channels_last)
scale_gpu = torch.randn(
    1, embedding_dim, 1, 1, device="cuda", dtype=dtype, requires_grad=True
).to(memory_format=torch.channels_last)
bias_gpu = torch.randn(
    1, embedding_dim, 1, 1, device="cuda", dtype=dtype, requires_grad=True
).to(memory_format=torch.channels_last)

# Epsilon must be a scalar value on the cpu.
epsilon_cpu = torch.full(
    (1, 1, 1, 1), epsilon_value, dtype=torch.float32, requires_grad=False, device="cpu"
)

Next, create the graph for the forward pass.

In [5]:
with cudnn.Graph(
    io_data_type=cudnn.data_type.FLOAT,
    intermediate_data_type=cudnn.data_type.FLOAT,
    compute_data_type=cudnn.data_type.FLOAT,
) as fwd_graph:
    # layernorm forward pass
    norm_out, mean, inv_var = fwd_graph.layernorm(
        name="LN",
        norm_forward_phase=cudnn.norm_forward_phase.TRAINING,
        input=x_gpu,
        scale=scale_gpu,
        bias=bias_gpu,
        epsilon=epsilon_cpu,
    )
    # relu on the layernorm output
    out = fwd_graph.relu(
        name="ReLU",
        input=norm_out,
        lower_clip=lower_clip_val,
        upper_clip=upper_clip_val,
    )
    # mark the output tensors
    out.set_name("output").set_output(True)
    mean.set_name("mean").set_output(True).set_data_type(cudnn.data_type.FLOAT)
    inv_var.set_name("inv_var").set_output(True).set_data_type(cudnn.data_type.FLOAT)

hi


Then, execute the graph and compare the output to the reference output from PyTorch:

In [6]:
# allocated output tensors
out_gpu = torch.empty_like(x_gpu)
mean_gpu = torch.empty(batch * seq_size, 1, 1, 1, dtype=torch.float32, device="cuda")
inv_var_gpu = torch.empty(batch * seq_size, 1, 1, 1, dtype=torch.float32, device="cuda")

# execute the graph
output = fwd_graph(
    {
        # input tensors
        "LN::input": x_gpu,
        "LN::scale": scale_gpu,
        "LN::bias": bias_gpu,
        "LN::epsilon": epsilon_cpu,
        # output tensors
        "output": out_gpu,
        "mean": mean_gpu,
        "inv_var": inv_var_gpu,
    },
    handle=handle,
)

# PyTorch reference forward operation
normalized_x = torch.nn.functional.layer_norm(
    x_gpu,
    [embedding_dim, 1, 1],
    weight=scale_gpu.squeeze(0),
    bias=bias_gpu.squeeze(0),
    eps=epsilon_value,
)
out_ref = torch.clamp(normalized_x, min=lower_clip_val, max=upper_clip_val)
mean_ref = x_gpu.to(torch.float32).mean(dim=(1, 2, 3), keepdim=True)
inv_var_ref = torch.rsqrt(
    torch.var(x_gpu.to(torch.float32), dim=(1, 2, 3), keepdim=True) + epsilon_value
)

# compare to reference output
torch.testing.assert_close(out_gpu, out_ref, rtol=5e-3, atol=5e-3)
torch.testing.assert_close(inv_var_gpu, inv_var_ref, rtol=5e-3, atol=5e-3)
torch.testing.assert_close(mean_gpu, mean_ref, rtol=5e-3, atol=5e-3)

#### LayerNorm ReLU Backward Pass

In the no bitmask implementation of dReLU, we determine dReLU activations by recomputing the y output with a prologue forward layer norm inference fusion.

In [7]:
# Reference backward operation using PyTorch
target = torch.randn_like(out_ref)
criterion = torch.nn.MSELoss()
loss = criterion(out_ref, target)

out_ref.retain_grad()
x_gpu.retain_grad()
scale_gpu.retain_grad()
bias_gpu.retain_grad()

loss.backward()

In [10]:
# Backward pass
with cudnn.Graph(
    intermediate_data_type=cudnn.data_type.FLOAT,
    compute_data_type=cudnn.data_type.FLOAT,
) as bwd_graph:
    # layernorm forward inference to re-compute forward y output
    y_out, _, _ = bwd_graph.layernorm(
        name="LN_infer",
        norm_forward_phase=cudnn.norm_forward_phase.INFERENCE,
        input=x_gpu,
        scale=scale_gpu,
        bias=bias_gpu,
        pre_calculated_mean=mean_gpu,
        pre_calculated_invvar=inv_var_gpu,
        epsilon=epsilon_cpu,
    )
    # dReLU computation using the re-computed y output
    drelu_dY = bwd_graph.relu_backward(
        name="DReLU",
        loss=out_ref.grad,
        input=y_out,
        lower_clip=lower_clip_val,
        upper_clip=upper_clip_val,
    )
    # the layernorm backward operation
    d_x, d_scale, d_bias = bwd_graph.layernorm_backward(
        name="DLN",
        grad=drelu_dY,
        input=x_gpu,
        scale=scale_gpu,
        mean=mean_gpu,
        inv_variance=inv_var_gpu,
    )
    # mark the output tensors
    d_x.set_output(True).set_name("dX").set_data_type(dtype)
    d_scale.set_output(True).set_name("dScale").set_data_type(dtype)
    d_bias.set_output(True).set_name("dBias").set_data_type(dtype)

# Execute the backward graph
output = bwd_graph(
    {
        # LN_infer inputs
        "LN_infer::input": x_gpu.detach(),
        "LN_infer::scale": scale_gpu.detach(),
        "LN_infer::bias": bias_gpu.detach(),
        "LN_infer::epsilon": epsilon_cpu,
        "LN_infer::pre_calculated_mean": mean_gpu.detach(),
        "LN_infer::pre_calculated_invvar": inv_var_gpu.detach(),
        # DReLU inputs (only loss, input is virtual from LN_infer)
        "DReLU::loss": out_ref.grad,
        # DLN inputs (grad is virtual from DReLU)
        "DLN::input": x_gpu.detach(),
        "DLN::scale": scale_gpu.detach(),
        "DLN::mean": mean_gpu.detach(),
        "DLN::inv_variance": inv_var_gpu.detach(),
        # Outputs
        "dX": torch.empty_like(x_gpu),
        "dScale": torch.empty_like(scale_gpu),
        "dBias": torch.empty_like(bias_gpu),
    },
    handle=handle,
)

# Extract outputs
d_x_gpu = output["dX"]
d_scale_gpu = output["dScale"]
d_bias_gpu = output["dBias"]

# compare to reference output
torch.testing.assert_close(x_gpu.grad, d_x_gpu, atol=2e-4, rtol=2e-4)
torch.testing.assert_close(scale_gpu.grad, d_scale_gpu, atol=2e-4, rtol=2e-4)
torch.testing.assert_close(bias_gpu.grad, d_bias_gpu, atol=2e-4, rtol=2e-4)

passed


## Using Python Binding APIs

#### LayerNorm ReLU Training Forward Pass

Create input tensor GPU buffers. We use PyTorch to allocate GPU tensors so we can reuse them easily when we calculate reference outputs.

In [11]:
# Allocate input tensor memory, initialize them to random numbers
x_gpu = torch.randn(
    batch * seq_size,
    embedding_dim,
    1,
    1,
    device="cuda",
    dtype=dtype,
    requires_grad=True,
).to(memory_format=torch.channels_last)
scale_gpu = torch.randn(
    1, embedding_dim, 1, 1, device="cuda", dtype=dtype, requires_grad=True
).to(memory_format=torch.channels_last)
bias_gpu = torch.randn(
    1, embedding_dim, 1, 1, device="cuda", dtype=dtype, requires_grad=True
).to(memory_format=torch.channels_last)

# Epsilon must be a scalar value on the cpu.
epsilon_cpu = torch.full(
    (1, 1, 1, 1), epsilon_value, dtype=torch.float32, requires_grad=False, device="cpu"
)

Then we create the graph for the forward pass.

In [13]:
# Create the cuDNN graph.
graph = cudnn.pygraph(
    handle=handle,
    io_data_type=cudnn.data_type.FLOAT,
    intermediate_data_type=cudnn.data_type.FLOAT,
    compute_data_type=cudnn.data_type.FLOAT,
)

# Create tensor handles with the graph API, assign UIDs.
x = graph.tensor_like(x_gpu.detach()).set_name("X")
scale = graph.tensor_like(scale_gpu.detach()).set_name("scale")
bias = graph.tensor_like(bias_gpu.detach()).set_name("bias")
epsilon = graph.tensor_like(epsilon_cpu).set_name("epsilon")

# Add a layernorm operation
norm_out, mean, inv_var = graph.layernorm(
    name="layernorm",
    norm_forward_phase=cudnn.norm_forward_phase.TRAINING,
    input=x,
    scale=scale,
    bias=bias,
    epsilon=epsilon,
)

# Add a relu operation
out = graph.relu(
    name="relu", input=norm_out, lower_clip=lower_clip_val, upper_clip=upper_clip_val
)

# Enable all outputs, by default outputs are disabled
out.set_name("output").set_output(True)
mean.set_name("mean").set_output(True).set_data_type(cudnn.data_type.FLOAT)
inv_var.set_name("inv_var").set_output(True).set_data_type(cudnn.data_type.FLOAT)

# print(graph)

# Build the graph
graph.build([cudnn.heur_mode.A, cudnn.heur_mode.FALLBACK])

pased


Here we assign UIDs for tensors. UIDs are a unique identifier that will allow us to provide a mapping from tensors created from cuDNN graph api calls, such as `graph.tensor_like()`, to the underlying device memory that will be used to store these tensors. Virtual tensors don't require explicit memory allocated for them, but non-vritual tensors like inputs or outputs will need to have UIDs assigned to them. 

Alternatively, one can use handles directly in the mapping, however using UIDs can be more convinient for caching of cuDNN graphs.

For each of our inputs {X, Scale, Bias, Epsilon} and our outputs {Out, Mean, Inverse Variance}, we allocate a UID. 

After validating and building a cuDNN graph,  we can now execute it. To do this, we have to provide input and output buffers. We do this by using the previously allocated UIDs to associate between tensor handles generated from the graph API, and their underlying memory. 

The desired input values need to be stored in these buffers before the `graph.execute` call. Because we have done a reference computation, we can simply reuse the buffers we have allocated via PyTorch.

Note that the EPISLON UID expects a cpu buffer.

In [14]:
# Allocate output tensor memory.
out_gpu = torch.empty_like(x_gpu)
mean_gpu = torch.empty(batch * seq_size, dtype=torch.float32, device="cuda")
inv_var_gpu = torch.empty(batch * seq_size, dtype=torch.float32, device="cuda")

# mapping of handles -> memory
variant_pack = {
    x: x_gpu,
    scale: scale_gpu,
    bias: bias_gpu,
    epsilon: epsilon_cpu,
    out: out_gpu,
    mean: mean_gpu,
    inv_var: inv_var_gpu,
}
workspace = torch.empty(graph.get_workspace_size(), device="cuda", dtype=torch.uint8)
graph.execute(variant_pack, workspace)
torch.cuda.synchronize()

Compute reference ouputs.

In [15]:
# Reference forward operation using PyTorch
normalized_x = torch.nn.functional.layer_norm(
    x_gpu,
    [embedding_dim, 1, 1],
    weight=scale_gpu.squeeze(0),
    bias=bias_gpu.squeeze(0),
    eps=epsilon_value,
)
out_ref = torch.clamp(normalized_x, min=lower_clip_val, max=upper_clip_val)
mean_ref = x_gpu.to(torch.float32).mean(dim=(1, 2, 3))
inv_var_ref = torch.rsqrt(
    torch.var(x_gpu.to(torch.float32), dim=(1, 2, 3)) + epsilon_value
)

Test cuDNN's output against PyTorch's and check correctness

In [16]:
# compare to reference output
torch.testing.assert_close(out_gpu, out_ref, rtol=5e-3, atol=5e-3)
torch.testing.assert_close(inv_var_gpu, inv_var_ref, rtol=5e-3, atol=5e-3)
torch.testing.assert_close(mean_gpu, mean_ref, rtol=5e-3, atol=5e-3)

#### LayerNorm ReLU Backward Pass


Compute references values for backward graph

In [17]:
# Reference backward operation using PyTorch
target = torch.randn_like(out_ref)
criterion = torch.nn.MSELoss()
loss = criterion(out_ref, target)

out_ref.retain_grad()
x_gpu.retain_grad()
scale_gpu.retain_grad()
bias_gpu.retain_grad()

loss.backward()

Create cuDNN graph and tensors

In [18]:
bwd_graph = cudnn.pygraph(
    handle=handle,
    intermediate_data_type=cudnn.data_type.FLOAT,
    compute_data_type=cudnn.data_type.FLOAT,
)

# Create tensors associated with the backwards graph. DO NOT reuse tensor handles from the forward graph.
d_out = bwd_graph.tensor(
    name="d_out", dim=x_gpu.size(), stride=x_gpu.stride(), data_type=x_gpu.dtype
)

x_bwd = bwd_graph.tensor_like(x, name="x")
scale_bwd = bwd_graph.tensor_like(scale, name="scale")
bias_bwd = bwd_graph.tensor_like(bias, name="bias")
mean_bwd = bwd_graph.tensor_like(mean, name="mean")
inv_var_bwd = bwd_graph.tensor_like(inv_var, name="inv_var")
epsilon_bwd = graph.tensor_like(epsilon_cpu).set_name("epsilon")


# layernorm forward inference to re-compute forward y output
y_out, _, _ = bwd_graph.layernorm(
    name="LN_infer",
    norm_forward_phase=cudnn.norm_forward_phase.INFERENCE,
    input=x_bwd,
    scale=scale_bwd,
    bias=bias_bwd,
    pre_calculated_mean=mean_bwd,
    pre_calculated_invvar=inv_var_bwd,
    epsilon=epsilon_bwd,
)

# dReLU computation using the re-computed y output
drelu_dY = bwd_graph.relu_backward(
    name="DReLU",
    loss=d_out,
    input=y_out,
    lower_clip=lower_clip_val,
    upper_clip=upper_clip_val,
)

# the layernorm backward operation
d_x, d_scale, d_bias = bwd_graph.layernorm_backward(
    name="DLN",
    grad=drelu_dY,
    input=x_bwd,
    scale=scale_bwd,
    mean=mean_bwd,
    inv_variance=inv_var_bwd,
)

# Enable outputs.
d_x.set_output(True).set_data_type(x_gpu.dtype)
d_scale.set_output(True).set_data_type(x_gpu.dtype)
d_bias.set_output(True).set_data_type(x_gpu.dtype)

# print(bwd_graph)

# Build the bwd_graph
bwd_graph.build([cudnn.heur_mode.A, cudnn.heur_mode.FALLBACK])

Execute the graph 

In [19]:
# Create output buffers for gradients
d_x_gpu = torch.empty_like(x_gpu)
d_scale_gpu = torch.empty_like(scale_gpu)
d_bias_gpu = torch.empty_like(bias_gpu)

workspace = torch.empty(
    bwd_graph.get_workspace_size(), device="cuda", dtype=torch.uint8
)

# For the inputs of the backwards graph (x_bwd, d_out, scale_bwd, mean_bwd, inv_var_bwd), we use the outputs of the forwards graph. For d_out we use pytorches autograd .grad functionality.
variant_pack = {
    x_bwd: x_gpu.detach(),
    scale_bwd: scale_gpu.detach(),
    bias_bwd: bias_gpu.detach(),
    epsilon_bwd: epsilon_cpu,
    d_out: out_ref.grad,
    mean_bwd: mean_gpu.detach(),
    inv_var_bwd: inv_var_gpu.detach(),
    d_x: d_x_gpu,
    d_scale: d_scale_gpu,
    d_bias: d_bias_gpu,
}
bwd_graph.execute(variant_pack, workspace)
torch.cuda.synchronize()

Test cuDNN's output against PyTorch's and check correctness

In [20]:
# compare to reference output
torch.testing.assert_close(x_gpu.grad, d_x_gpu, atol=2e-4, rtol=2e-4)
torch.testing.assert_close(scale_gpu.grad, d_scale_gpu, atol=2e-4, rtol=2e-4)
torch.testing.assert_close(bias_gpu.grad, d_bias_gpu, atol=2e-4, rtol=2e-4)

passed


Perform Cleanup

In [ ]:
cudnn.destroy_handle(handle)